In [2]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import sympy as sp
import xarray as xr
from IPython.display import display,Math
warnings.filterwarnings('ignore')

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
KERNELFROM = CONFIGS['experiments']['sr']['runs']['sr_atm']['weightsfrom']
NNSEEDS    = CONFIGS['experiments']['nn']['seeds']
SRSEEDS    = CONFIGS['experiments']['sr']['seeds']
EQUATIONS  = CONFIGS['experiments']['sr']['optimizedeqs']
SPLIT      = 'test'
LANDFRAC   = 0.5
MINCOMPLEXITY = 8
RUNS       = {'sr_sfc':('sr_sfc_eq',[72,102]),'sr_all':('sr_all_eq',[102])}
REFERENCES = ['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']
NUMPYFUNCS = {'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,'abs':np.abs,'exp':np.exp,'max':np.maximum,'min':np.minimum}
SYMPYFUNCS = {'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,'abs':sp.Abs,'exp':sp.exp,'max':sp.Max,'min':sp.Min}
REGISTRY   = {row['name']:(row['form'],json.loads(row['constants'])) for _,row in pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv')).iterrows()}

In [4]:
def kernel_integrate(fields,weights,dsig):
    return (fields*weights[None,:,:]*dsig[None,None,:]).sum(axis=2)

def prepare_form(form):
    return re.sub(r'\^','**',form)

def evaluate(form,columns,constants=None):
    namespace = dict(NUMPYFUNCS,__builtins__={})
    namespace.update(columns)
    namespace.update(constants or {})
    out = eval(prepare_form(form),namespace)
    return np.broadcast_to(np.asarray(out,dtype=float),columns['lf'].shape)

def to_precip(z):
    return np.expm1(STATS['tp_std']*np.maximum(z,0.0))

def calc_r2(pred,target,mask):
    return 1-np.sum((pred[mask]-target[mask])**2)/np.sum((target[mask]-target[mask].mean())**2)

def calc_skill(z):
    pred = to_precip(z)
    return {f'R2 {region}':calc_r2(pred,tp,mask) for region,mask in regionmasks.items()}

def to_physical(form,constants=None):
    expr = sp.sympify(prepare_form(form),locals={**SYMPYFUNCS,**PHYSICAL,**{name:sp.Float(value) for name,value in (constants or {}).items()}})
    return sp.N(STATS['tp_std']*expr,4)

In [5]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    dsig    = ds['dsig'].values
    fields  = np.stack([ds[name].transpose('time','lat','lon','sig').values.reshape(-1,dsig.size) for name in FIELDVARS],axis=1)
    columns = {name:ds[name].transpose('time','lat','lon').values.ravel().astype(float) for name in ['shf','lhf']}
    columns['lf'] = xr.broadcast(ds['lf'],ds['shf'])[0].transpose('time','lat','lon').values.ravel().astype(float)
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    tp = ds['tp'].transpose('time','lat','lon').values.ravel().astype(float)

features = []
for seed in NNSEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'{KERNELFROM}_{seed}_weights.nc'),engine='h5netcdf') as ds:
        features.append(kernel_integrate(fields,ds['k'].values,dsig))
columns.update(dict(zip(FIELDVARS,np.mean(features,axis=0).T)))
columns['sr_atm_eq'] = evaluate(*REGISTRY['sr_atm_eq'][:1],columns,REGISTRY['sr_atm_eq'][1])
finite      = np.isfinite(tp)&np.all([np.isfinite(values) for values in columns.values()],axis=0)
regionmasks = {'Ocean':finite&(columns['lf']<LANDFRAC),'Land':finite&(columns['lf']>=LANDFRAC),'All':finite}

atmsymbol = sp.Symbol(r'\mathcal{A}')
PHYSICAL  = {'lf':sp.Symbol(r'\mathrm{LF}'),'sr_atm_eq':atmsymbol/STATS['tp_std']}
for name,label in [('rh',r'\widehat{\mathrm{RH}}'),('thetae',r'\widehat{\theta_e}'),('thetaestar',r'\widehat{\theta_e^*}'),('shf',r'\mathrm{SHF}'),('lhf',r'\mathrm{LHF}')]:
    PHYSICAL[name] = (sp.Symbol(label)-STATS[f'{name}_mean'])/STATS[f'{name}_std']

In [6]:
rows = []
for name in REFERENCES:
    form,constants = REGISTRY[name]
    rows.append({'Source':'optimized','Run':EQUATIONS[name]['description'],'Seed':'--','Complexity':np.nan,'CSV Loss':np.nan,'Reference Loss':np.nan,
                 **calc_skill(evaluate(form,columns,constants)),'Equation':form,'Constants':constants})
for run,(eqname,refseeds) in RUNS.items():
    for seed in SRSEEDS:
        equations = pd.read_csv(os.path.join(MODELSDIR,'sr',f'{run}_{seed}_equations.csv'))
        refloss   = equations.loc[equations['complexity']==EQUATIONS[eqname]['refcomplexity'],'loss'].iloc[0] if seed in refseeds else np.nan
        for _,row in equations[equations['complexity']>=MINCOMPLEXITY].iterrows():
            rows.append({'Source':'pysr','Run':run,'Seed':seed,'Complexity':row['complexity'],'CSV Loss':row['loss'],'Reference Loss':refloss,
                         **calc_skill(evaluate(row['equation'],columns)),'Equation':row['equation'],'Constants':None})
candidates = pd.DataFrame(rows)
candidates['Loss <= Reference'] = np.where(candidates['Reference Loss'].isna(),'--',np.where(candidates['CSV Loss']<=candidates['Reference Loss'],'yes','no'))
display(candidates.drop(columns='Constants').sort_values('R2 All',ascending=False).style.hide(axis='index').format(precision=3))

Source,Run,Seed,Complexity,CSV Loss,Reference Loss,R2 Ocean,R2 Land,R2 All,Equation,Loss <= Reference
pysr,sr_all,102,15.000,0.384,0.379,0.514,0.306,0.424,sr_atm_eq + ((thetae - (shf * -5.6251073)) * cube(0.6989902 - lf)),no
optimized,SR-ALL-PC,--,nan,nan,nan,0.509,0.294,0.416,sr_atm_eq+(thetae+c12*shf)*cube(c13-lf)+cube(1-c13)*thetae-c14,--
optimized,SR-ALL,--,nan,nan,nan,0.509,0.293,0.415,sr_atm_eq+(thetae+c9*shf)*cube(c10-lf)-c11,--
pysr,sr_all,102,17.000,0.379,0.379,0.507,0.292,0.414,(sr_atm_eq + ((thetae - (shf * -5.243847)) * cube(0.7261934 - lf))) + -0.123223685,yes
pysr,sr_all,102,19.000,0.376,0.379,0.503,0.276,0.404,(sr_atm_eq * 0.9045838) + ((shf + (thetae * 0.19646363)) * square(cube(1.1292378 - lf))),yes
pysr,sr_all,102,14.000,0.390,0.379,0.470,0.302,0.398,sr_atm_eq + ((shf - -0.08833005) * cube(square(1.1091791 - lf))),no
pysr,sr_sfc,102,18.000,0.378,0.392,0.472,0.297,0.397,"min(((shf * (0.8336486 - lf)) + 0.07292696) * 2.285236, lhf + 1.0954965) + sr_atm_eq",yes
pysr,sr_sfc,72,18.000,0.372,0.388,0.471,0.296,0.396,"sr_atm_eq + (min(lhf, ((shf * (0.7948719 - lf)) * 2.2753685) + -0.89431965) - -1.0714867)",yes
pysr,sr_all,42,20.000,0.367,nan,0.466,0.300,0.395,"((min(shf, sr_atm_eq) / -0.44202757) * (lf + min(-0.7758296, lhf))) + (sr_atm_eq + 0.1570777)",--
pysr,sr_all,42,19.000,0.367,nan,0.464,0.300,0.394,"((min(shf, sr_atm_eq) * -2.21411) * (lf + min(-0.79319876, lhf))) + (sr_atm_eq + 0.14164433)",--


In [7]:
print('Physical exponent E, where P = (exp(E)-1)^+ and A = lambda*max(M,I)^3 is the SR-ATM exponent')
selected = candidates[(candidates['Source']=='optimized')|(candidates['Loss <= Reference']=='yes')|(candidates['R2 All']>=candidates.loc[candidates['Run']=='SR-ALL-PC','R2 All'].iloc[0])]
for _,row in selected.sort_values('R2 All',ascending=False).iterrows():
    print(f"{row['Run']} | seed {row['Seed']} | complexity {row['Complexity']} | R2 all {row['R2 All']:.3f}, ocean {row['R2 Ocean']:.3f}, land {row['R2 Land']:.3f}")
    display(Math(r'\mathcal{E} = '+sp.latex(to_physical(row['Equation'],row['Constants']))))

Physical exponent E, where P = (exp(E)-1)^+ and A = lambda*max(M,I)^3 is the SR-ATM exponent
sr_all | seed 102 | complexity 15.0 | R2 all 0.424, ocean 0.514, land 0.306


<IPython.core.display.Math object>

SR-ALL-PC | seed -- | complexity nan | R2 all 0.416, ocean 0.509, land 0.294


<IPython.core.display.Math object>

SR-ALL | seed -- | complexity nan | R2 all 0.415, ocean 0.509, land 0.293


...

<IPython.core.display.Math object>

sr_sfc | seed 72 | complexity 17.0 | R2 all 0.003, ocean -0.249, land 0.287


<IPython.core.display.Math object>

sr_sfc | seed 102 | complexity 15.0 | R2 all -0.034, ocean -0.326, land 0.298


<IPython.core.display.Math object>